:)

In [ ]:
import pandas as pd
import numpy as np
import pickle
from scipy.optimize import linprog
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from collections import defaultdict
import matplotlib.patches as mpatches  # Needed for legend

In [ ]:
home_path = '/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/'
SPECIES = 'segal_species' # 'mpa_species' or 'segal_species'
PROBLEM = 'regression'

figures_path = '/net/mraid20/ifs/wisdom/segal_lab/genie/LabData/Analyses/tomerse/diet_mb/figures/diet_intervention'

Load Data

In [ ]:
phenotypes_mb = pd.read_pickle('/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/phenotypes_mb.pkl')
phenotype_df = phenotypes_mb[['bt__triglycerides']]
phenotype_df

In [ ]:
species = '' if SPECIES == 'segal_species' else '_mpa'

diet_mb = pd.read_pickle(home_path + f"data/{SPECIES}/diet_mb.pkl")
with open(home_path + f'data/{SPECIES}/my_lists.pkl', 'rb') as file:
    loaded_lists = pickle.load(file)
base_features, all_features, targets = loaded_lists
with open(home_path + f'data/{SPECIES}/scaler.pkl', 'rb') as scaler_file:
        scaler = pickle.load(scaler_file)
diet_mb

# Filter non significant correlations from the permutations
with open(f'/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/{PROBLEM}/{SPECIES}/significant_targets.pkl', 'rb') as file:
    loaded_lists = pickle.load(file)
significant_targets = loaded_lists
significant_targets_indices = [targets.index(item) for item in significant_targets]

with open('/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/food_shortnames.pkl', 'rb') as file:
    food_shortnames = pickle.load(file)
food_shortnames

In [ ]:
significant_targets

In [ ]:
diet_mb[all_features]

In [ ]:
diet_mb[targets]

In [ ]:
diet_mb.iloc[3903]

In [ ]:
significant_targets_df = pd.DataFrame(columns=significant_targets)
significant_targets_df

In [ ]:
mb_names = pd.read_pickle("/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/mb_names.pkl")

def rename_microbiome_columns(df: pd.DataFrame, mb_names: pd.DataFrame) -> pd.DataFrame:
    """
    Get either microbial_features (from diet_mb) or gut_bacteria_df (from the loader)
    """
    # --- Normalize ---
    mb_names.index = mb_names.index.str.strip()
    df = df.copy()
    df.columns = df.columns.str.strip()

    # --- Extract maps ---
    species_map = mb_names['species_new'].str.strip()
    genus_map = mb_names['genus_new'].str.strip()
    family_map = mb_names['family_new'].str.strip()

    # --- Build name mapping ---
    final_mapping = {}
    for col in df.columns:  # Skip RegistrationCode or first identifier column
        name = species_map.get(col, None)
        if name == "unknown" or pd.isna(name):
            name = genus_map.get(col, None)
        if name == "unknown" or pd.isna(name):
            name = family_map.get(col, None)
        if name is None or name == "unknown":
            name = col  # fallback
        final_mapping[col] = name

    # --- Rename columns ---
    df.rename(columns=final_mapping, inplace=True)

    # --- Deduplicate ---
    col_counts = Counter(df.columns)
    name_counter = defaultdict(int)
    new_cols = []

    for col in df.columns:
        if col_counts[col] > 1:
            name_counter[col] += 1
            new_cols.append(f"{col}_{name_counter[col]}")
        else:
            new_cols.append(col)

    df.columns = new_cols

    return df

bacterial_features_df = rename_microbiome_columns(diet_mb[targets], mb_names)
bacterial_features_df

In [ ]:
significant_targets_df = rename_microbiome_columns(significant_targets_df, mb_names)
significant_targets_df

"Bifidobacterium longum" in significant_targets_df.columns

In [ ]:
import pandas as pd
from collections import Counter, defaultdict

def rename_microbiome_series(series_data: pd.Series, mb_names: pd.DataFrame) -> pd.Series:
    """
    Renames the index of a pandas Series based on a provided microbiome mapping.
    """
        # --- Input Validation and Setup ---
    if not isinstance(series_data, pd.Series):
        raise TypeError("Input 'series_data' must be a pandas Series.")
    
    # --- Normalize ---
    mb_names.index = mb_names.index.str.strip()
    data_copy = series_data.copy()
    data_copy.index = data_copy.index.str.strip()

    # --- Extract maps ---
    species_map = mb_names['species_new'].str.strip()
    genus_map = mb_names['genus_new'].str.strip()
    family_map = mb_names['family_new'].str.strip()

    # --- Build name mapping ---
    final_mapping = {}
    for original_name in data_copy.index:
        name = species_map.get(original_name, None)
        if name == "unknown" or pd.isna(name):
            name = genus_map.get(original_name, None)
        if name == "unknown" or pd.isna(name):
            name = family_map.get(original_name, None)
        if name is None or name == "unknown":
            name = original_name  # fallback
        final_mapping[original_name] = name

    # --- Rename index ---
    data_copy.rename(index=final_mapping, inplace=True)

    # --- Deduplicate ---
    idx_counts = Counter(data_copy.index)
    name_counter = defaultdict(int)
    new_idx = []

    for idx_val in data_copy.index:
        if idx_counts[idx_val] > 1:
            name_counter[idx_val] += 1
            new_idx.append(f"{idx_val}_{name_counter[idx_val]}")
        else:
            new_idx.append(idx_val)

    data_copy.index = new_idx

    return data_copy

In [ ]:
(diet_mb['Wine'] > 0).sum()

In [ ]:
directional_phenotypes_shap = pd.read_pickle('/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/diet_intervention/directional_phenotypes_shap.pkl')

# Remove age and gender
directional_phenotypes_shap = directional_phenotypes_shap[~directional_phenotypes_shap.index.isin(['age', 'sex'])]
directional_phenotypes_shap = directional_phenotypes_shap.iloc[significant_targets_indices, :]
directional_phenotypes_shap

In [ ]:
directional_phenotypes_shap['bt__triglycerides'].sort_values()

In [ ]:
directional_microbiome_shap = pd.read_pickle('/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/diet_intervention/directional_microbiome_shap.pkl')

# Remove age and gender
directional_microbiome_shap = directional_microbiome_shap[~directional_microbiome_shap.index.isin(['age', 'sex'])]
directional_microbiome_shap

In [ ]:
directional_microbiome_shap.loc["Dates", :].abs().sort_values(ascending=False).head(20)

In [ ]:
def get_subject(diet_mb, i):
    subject = diet_mb.iloc[i]

    subject_gender = subject['sex']
    subject_age = subject['age']

    subject_diet = subject[all_features]
    subject_diet = subject_diet[~subject_diet.index.isin(['age', 'sex'])]

    subject_calories = subject_diet['Energy']

    subject_diet = subject_diet[food_shortnames]

    # subject_mb = subject[significant_targets]
    subject_mb = subject[targets]
    # subject_mb.index = phenotype_shap.index
    subject_mb.index = targets
    subject_mb = 10** subject_mb

    return subject_diet, subject_mb, subject_calories, subject_gender, subject_age

In [ ]:
# indices_to_find = [
#     "Parabacteroides distasonis",
#     "Fusicatenibacter saccharivorans",
#     "Faecalibacterium prausnitzii_D",
#     "Bacteroides uniformis",
#     "Phocaeicola vulgatus"
# ]

# for index_to_find in indices_to_find:
#     location = directional_triglycerides_shap.index.get_loc("Bifidobacterium longum")
#     print(f"Index '{index_to_find}' is at location: {location}")

In [ ]:
directional_triglycerides_shap = directional_phenotypes_shap.loc[:, 'bt__triglycerides']
# directional_triglycerides_shap[29] = directional_triglycerides_shap[29] * 1000000 # For testing!
directional_triglycerides_shap.abs().sort_values(ascending=False)

In [ ]:
triglyceride_species = [
    "Otoolea fessa",
    "Lachnospira pectinoschiza_A",
    "Bifidobacterium longum",
    "Alistipescatomonas sp900066785",
    "Faecalibacterium longum_1",
    "Ligilactobacillus ruminis",
    "Acetatifactor intestinalis_1",
    "Choladosuia sp902363665",
    "Agathobacter rectalis",
    "Enterocloster sp000431375",
    "Roseburia intestinalis",
    "Bacteroides cellulosilyticus",
    "UBA11524 sp000437595",
    "Streptococcus parasanguinis",
    "Lachnospira hominis",
    "Klebsiella pneumoniae",
    "Merdivicinus sp934539585",
    "Faecalibacterium sp900539885"
]

In [ ]:
directional_triglycerides_shap[[639, 251, 466, 625, 615]]

In [ ]:
directional_microbiome_shap = directional_microbiome_shap.loc[food_shortnames, :]

In [ ]:
print(directional_microbiome_shap.shape)
print(directional_triglycerides_shap.shape)
# print(subject_diet.shape)
# print(subject_mb.shape)

In [ ]:
diet_mb["Dates"].sort_values()

In [ ]:
prevalence_count = [(diet_mb[target] > -4).sum() for target in significant_targets]
prevalence_count = pd.Series(prevalence_count, index=directional_phenotypes_shap.index)
prevalence_count.sort_values()

### TODO's

In [ ]:
#TODO output the expected effect on triglicerydies / bacteria from each food change 
# Step 1 : Filter subjects that have an actuall value in the target phenotype and not just NA

#TODO Check the 20% caloric restriction, how does it effect the model

#TODO Any other constraints ?

diet_foods_df = pd.read_csv('/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/data/diet_adherence_foods.csv', index_col=0)
diet_foods_df = diet_foods_df.loc[:,diet_foods_df.columns.isin(food_shortnames)]
nova_foods_df = diet_foods_df.loc['NOVA', :]
nova_foods_df

# Food shortnames is the list of foods, you should already have this loaded.

In [ ]:
positive_diets = diet_mb[diet_mb > 0]

# No worries that there are also non food names here (like bacteria), will be filtered inside the function
food_increase_upper_bounds = positive_diets.quantile(0.95).fillna(0.0)
food_increase_upper_bounds

In [ ]:
food_increase_upper_bounds['Milk']

Most updated code

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="Trying to unpickle estimator StandardScaler")

# --- Define the recommendation function ---
def recommend_diet_change(
    diet_species_shap, species_health_shap, current_diet_pct, current_species_abundance, bacterial_features_df,
    food_increase_upper_bounds, nova_foods_df, max_pct_change, subject_id, subject_calories, subject_gender):

    # print("--- Starting Personalized Dietary Recommendation ---")

    # --- Data Alignment ---
    # print("--- Aligning Data ---")
    common_species = diet_species_shap.columns.intersection(species_health_shap.index).intersection(current_species_abundance.index)
    common_foods = diet_species_shap.index.intersection(current_diet_pct.index)

    if len(common_species) == 0 or len(common_foods) == 0:
        raise ValueError("No common species or foods found between the provided data. Please check your labels.")

    # Align data
    diet_species_shap_aligned = diet_species_shap.loc[common_foods, common_species]
    species_health_shap_aligned = species_health_shap.loc[common_species]

    current_diet_pct_aligned = current_diet_pct.loc[common_foods]
    current_species_abundance_aligned = current_species_abundance.loc[common_species]

    # --- Filter out foods with <5 kcal/day ---
    min_kcal_threshold = 5
    valid_foods_mask = (current_diet_pct_aligned * subject_calories) >= min_kcal_threshold
    current_diet_pct_aligned = current_diet_pct_aligned[valid_foods_mask]
    diet_species_shap_aligned = diet_species_shap_aligned.loc[current_diet_pct_aligned.index]

    if len(current_diet_pct_aligned) == 0:
        print(f"Subject {subject_id} has no foods meeting the minimum kcal threshold. Skipping.")
        return None

    if current_diet_pct_aligned.sum() == 0:
        print(f"Subject {subject_id} has no current intake data (all zeros). Skipping.")
        return None

    if current_diet_pct_aligned.isnull().any() or (current_diet_pct_aligned < 0).any():
        raise ValueError("Current diet intake contains NaNs or negative values, check your data.")

    num_foods = len(current_diet_pct_aligned)
    epsilon = 1e-9

    # --- Personalization weighting ---
    ####
    # personalization_weights = 1 / (current_species_abundance_aligned + epsilon)
    # # personalization_weights = current_species_abundance_aligned + epsilon
    # personalization_weights /= personalization_weights.mean()
    # personalized_diet_species_shap = diet_species_shap_aligned.multiply(personalization_weights, axis='columns')
    personalized_diet_species_shap = diet_species_shap_aligned
    ####

    # --- Compute personalized food impact scores ---
    food_impact_scores = personalized_diet_species_shap.dot(species_health_shap_aligned)

    # --- Optimization setup ---
    c = np.concatenate([food_impact_scores, -food_impact_scores])

    # --- Calorie change constraints ---
    A_net_cal_change = np.concatenate([np.ones(num_foods), -np.ones(num_foods)])
    net_cal_limit = max_pct_change * current_diet_pct_aligned.sum()

    A_ub = [
        np.ones(num_foods * 2),      # Total % change constraint
        A_net_cal_change,            # Net +/− calorie constraint
        -A_net_cal_change
    ]
    b_ub = [
        max_pct_change,
        net_cal_limit,
        net_cal_limit
    ]

    # --- Alcohol constraint setup ---
    alcoholic_features = [
        "Beer", "Campari", "Cocktail", "Dessert Wine", "Gin and tonic", "Light Beer",
        "Malt beverage", "Ouzo", "Sweet wine", "Vodka or Arak", "Whiskey", "Wine",
    ]

    # Woman 0.5-1.5 standard drink a day which is 70-105kcal (AHA, Harvard, USDA)
    # Men 0.5-2 drink which is 70-140 kcal

    if subject_gender == 1:
        max_alcohol_kcal = 0.07 * subject_calories  # Male: ≤7% of total kcal
    elif subject_gender == 0:
        max_alcohol_kcal = 0.05 * subject_calories  # Female: ≤5%
    else:
        max_alcohol_kcal = 0.05 * subject_calories  # Conservative fallback

    # --- Build bounds ---
    current_bounds = []
    alcohol_indexes = []

    for i, (food_name, food_pct_val) in enumerate(current_diet_pct_aligned.items()):
        nova_score = nova_foods_df.get(food_name, None)
        if nova_score == 4:
            increase_bound = 0
        elif food_name in alcoholic_features:
            if food_pct_val > 0:
                increase_bound = food_pct_val
                alcohol_indexes.append(i)
            else:
                increase_bound = 0
        elif food_pct_val > 0: # Offer only foods present in the diet
        # elif food_pct_val > 0 or food_pct_val <= 0:
            max_pct_limit = food_increase_upper_bounds.get(food_name, 0)
            increase_bound = max(0, max_pct_limit - food_pct_val)
        else:
            increase_bound = 0

        decrease_bound = food_pct_val if food_pct_val > 0 else 0

        current_bounds.append((0, increase_bound))
        current_bounds.append((0, decrease_bound))

    # --- Alcohol intake constraint (total kcal increase from alcohols) ---
    if alcohol_indexes:
        alcohol_constraint = np.zeros(num_foods * 2)
        for idx in alcohol_indexes:
            # Final value = original + (increase - decrease)
            alcohol_constraint[idx] = subject_calories  # increase part
            alcohol_constraint[num_foods + idx] = -subject_calories  # subtract decrease
            # final = original + increase - decrease

        total_starting_alcohol_kcal = sum(
            current_diet_pct_aligned.iloc[idx] * subject_calories for idx in alcohol_indexes
        )

        remaining_alcohol_kcal = max_alcohol_kcal - total_starting_alcohol_kcal

        # If the starting amount is already over the limit, disallow any increases
        if remaining_alcohol_kcal < 0:
            remaining_alcohol_kcal = 0

        A_ub.append(alcohol_constraint)
        b_ub.append(remaining_alcohol_kcal)

    # print(f"Subject {subject_id}: Total foods allowed to change before filtration: {sum((ub-lb)>1e-9 for lb, ub in current_bounds)}")

    result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=current_bounds, method='highs', options={"disp": False})

    if not result.success:
        print("Could not find a valid solution. Returning None.")
        return None

    # print("Optimization successful!")

    solution = result.x
    d_increase_pct = solution[:num_foods]
    d_decrease_pct = solution[num_foods:]
    net_change_pct = d_increase_pct - d_decrease_pct

    # print(f"net_change_pct: min {net_change_pct.min():.6f}, max {net_change_pct.max():.6f}")


    recommendation = pd.DataFrame({
        'Current Diet (% Calorie)': current_diet_pct_aligned,
        'Recommended Change (% Calorie)': net_change_pct,
        'Personalized Impact Score': food_impact_scores,
    })
    recommendation['New Diet (% Calorie)'] = recommendation['Current Diet (% Calorie)'] + recommendation['Recommended Change (% Calorie)']
    recommendation['New Diet (% Calorie)'] = recommendation['New Diet (% Calorie)'].clip(lower=0)
    recommendation['Absolute Predicted Impact'] = (recommendation['Recommended Change (% Calorie)'] * recommendation['Personalized Impact Score']).abs()

    recommendation['Current Diet (kcal)'] = (recommendation['Current Diet (% Calorie)'] * subject_calories).round(1)
    recommendation['Recommended Change (kcal)'] = (recommendation['Recommended Change (% Calorie)'] * subject_calories).round(1)
    recommendation['New Diet (kcal)'] = (recommendation['New Diet (% Calorie)'] * subject_calories).round(1)

    # Compute predicted change in microbial species via SHAP-weighted food changes

    # clip turns negative recommendations into 0
    updated_diet_pct = (current_diet_pct_aligned + net_change_pct).clip(lower=0) 
 
        # Load scaler and get expected feature order
    with open(home_path + f'data/{SPECIES}/diet_scaler.pkl', 'rb') as f:
        diet_scaler = pickle.load(f)

    ###
    with open(home_path + f'data/{SPECIES}/age_scaler.pkl', 'rb') as f:
        age_scaler = pickle.load(f)
        
    full_diet_index = full_diet_index = [col for col in diet_mb[all_features].columns if col != 'sex']

    # Create full vector with zeros
    full_diet_vector = pd.Series(0.0, index=full_diet_index)

    # Fill in current subject’s values
    full_diet_vector.loc[updated_diet_pct.index] = updated_diet_pct

    # --- Start of Modified Section ---
    
    # Initialize a new series to hold the scaled values
    updated_diet_z_full = pd.Series(index=full_diet_index, dtype=float)

    # 1. Scale 'age' using the age_scaler
    # Reshape is used because the scaler expects a 2D array
    age_value = full_diet_vector[['age']].values.reshape(1, -1)
    updated_diet_z_full['age'] = age_scaler.transform(age_value).flatten()[0]

    # 2. Scale the remaining diet features using the diet_scaler
    other_features_index = [col for col in full_diet_index if col != 'age']
    if other_features_index: # Proceed only if there are other features to scale
        other_features_values = full_diet_vector[other_features_index].values.reshape(1, -1)
        scaled_other_features = diet_scaler.transform(other_features_values).flatten()
        updated_diet_z_full.loc[other_features_index] = scaled_other_features
    ###

    # # Z-transform using the full vector
    # updated_diet_z_full = pd.Series(
    #     diet_scaler.transform(full_diet_vector.values.reshape(1, -1)).flatten(),
    #     index=full_diet_index
    # )

    # Extract just the z-scored values of the foods used by the subject
    updated_diet_z = updated_diet_z_full.loc[updated_diet_pct.index]

    delta_species_log_z = updated_diet_z @ diet_species_shap_aligned  

    # Map delta_species_log_z into full 724-species space
    delta_species_full_log_z = pd.Series(0.0, index=bacterial_features_df.columns)  
    delta_species_full_log_z.loc[delta_species_log_z.index] = delta_species_log_z.values

    with open(home_path + f'data/{SPECIES}/mb_scaler.pkl', 'rb') as f:
        species_scaler = pickle.load(f)

    # Get baseline log10(abundance)
    log_baseline = np.log10(current_species_abundance_aligned + epsilon)

    # Build full log10 vector for all species
    log_baseline_full = pd.Series(0.0, index=delta_species_full_log_z.index)
    log_baseline_full.loc[log_baseline.index] = log_baseline

    # --- Step 2: Transform baseline to z-space ---
    log_baseline_z = pd.Series(
        species_scaler.transform(log_baseline_full.values.reshape(1, -1)).flatten(),
        index=delta_species_full_log_z.index
    )

    # --- Step 3: Add SHAP delta in z-space ---
    log_predicted_z = log_baseline_z + delta_species_full_log_z

    # --- Step 4: Inverse-transform back to log10(abundance) ---
    log_predicted = pd.Series(
        species_scaler.inverse_transform(log_predicted_z.values.reshape(1, -1)).flatten(),
        index=delta_species_full_log_z.index
    )

    # Focus only on the species originally aligned

    # Tomer! For downstream
    # ----------------------------------------------------------
    log_predicted = log_predicted.loc[delta_species_log_z.index]
    # ----------------------------------------------------------

    # --- Optionally: predicted abundance in linear space ---
    predicted_abundance = 10 ** log_predicted
    
    # Get baseline abundance in linear space (optional, but useful)
    baseline_abundance = current_species_abundance_aligned.loc[log_predicted.index]

    fold_change = predicted_abundance / baseline_abundance

    delta_species_linear = predicted_abundance - baseline_abundance

    # Create tidy DataFrame for downstream plotting
    species_df = pd.DataFrame({
        'Species': log_predicted.index,
        'Baseline Abundance': baseline_abundance.values,
        'Predicted Δ fold Abundance': fold_change.values,
        'Predicted Δ linear Abundance': delta_species_linear.values,
        'Predicted Abundance': predicted_abundance.values,
        'Subject': subject_id
    })

    recommendation = recommendation[[
        'Current Diet (kcal)',
        'Recommended Change (kcal)',
        'New Diet (kcal)',
        'Current Diet (% Calorie)',
        'Recommended Change (% Calorie)',
        'New Diet (% Calorie)',
        'Personalized Impact Score',
        'Absolute Predicted Impact'
    ]]

    return recommendation, species_df

# --- Generate Recommendations and Visualizations ---
all_recommendations = []
microbial_shifts = []
subject_abundances = {}  
food_change_records = defaultdict(list)
all_mb_predictions = []
subject_ages = []
subject_genders = []
food_impact_scores_list = []

for i in range(100):
# for i in range(len(diet_mb.index)):
    print(i)
    subject_diet, subject_mb, subject_calories, subject_gender, subject_age = get_subject(diet_mb, i)
    ###
    subject_baseline_mb = subject_mb.copy()
    subject_baseline_mb = rename_microbiome_series(subject_baseline_mb, mb_names)
    # print(subject_baseline_mb)
    # registration_code = diet_mb.iloc[i].index
    # print(registration_code)
    subject_mb = subject_mb[significant_targets]
    # subject_mb.index = directional_triglycerides_shap.index
    subject_mb = rename_microbiome_series(subject_baseline_mb, mb_names)
    ###  
    subject_mb = subject_mb[subject_mb > 0.0001]

    output = recommend_diet_change(
        diet_species_shap=directional_microbiome_shap,
        species_health_shap=directional_triglycerides_shap,
        current_diet_pct=subject_diet,
        current_species_abundance=subject_mb,
        bacterial_features_df=bacterial_features_df,
        subject_calories=subject_calories,
        subject_gender=subject_gender,
        subject_id=i,
        food_increase_upper_bounds=food_increase_upper_bounds,
        nova_foods_df=nova_foods_df,
        max_pct_change=0.2
    )

    # Skip weird people
    if output is None:
        print(f"Skipping subject {i} due to optimization failure or invalid input.")
        continue

    recommendation, delta_species_df = output

    food_impact_scores_list.append(recommendation['Personalized Impact Score'])

    ####

    # print("delta_species_df:")
    # print(delta_species_df)
    subject_predicted_mb_present = delta_species_df.set_index('Species')['Predicted Abundance']
    # print("subject_predicted_mb_present")
    # print(subject_predicted_mb_present)
    subject_predicted_mb = subject_baseline_mb.copy()

    subject_predicted_mb.update(subject_predicted_mb_present)

    #  EXPERIMENT: artificially increase/decrease the RA of high impact species
    # if subject_predicted_mb["Bifidobacterium longum"] > 0.0001:
    #     subject_predicted_mb["Bifidobacterium longum"] = subject_predicted_mb["Bifidobacterium longum"] * 10
    # if subject_predicted_mb["Otoolea fessa"] > 0.0001:
    #     subject_predicted_mb["Otoolea fessa"] = max(subject_predicted_mb["Otoolea fessa"] / 10, 0.0001)
    # if subject_predicted_mb["Lachnospira pectinoschiza_A"] > 0.0001:
    #     subject_predicted_mb["Lachnospira pectinoschiza_A"] = max(subject_predicted_mb["Lachnospira pectinoschiza_A"] / 10, 0.0001)
    # if subject_predicted_mb["Faecalibacterium longum_1"] > 0.0001:
    #     subject_predicted_mb["Faecalibacterium longum_1"] = subject_predicted_mb["Faecalibacterium longum_1"] * 10
    # if subject_predicted_mb["UBA11524 sp000437595"] > 0.0001:
    #     subject_predicted_mb["UBA11524 sp000437595"] = subject_predicted_mb["UBA11524 sp000437595"] * 10
    # if subject_predicted_mb["Agathobacter rectalis"] > 0.0001:
    #     subject_predicted_mb["Agathobacter rectalis"] = max(subject_predicted_mb["Agathobacter rectalis"] / 10, 0.0001)

        #         triglyceride_species = [
#     "Ligilactobacillus ruminis",
#     "Acetatifactor intestinalis_1",
#     "Choladosuia sp902363665",
#     "Agathobacter rectalis",
#     "Enterocloster sp000431375",
#     "Roseburia intestinalis",
#     "Bacteroides cellulosilyticus",
#     "UBA11524 sp000437595",
#     "Streptococcus parasanguinis",
#     "Lachnospira hominis",
#     "Klebsiella pneumoniae",
#     "Merdivicinus sp934539585",
#     "Faecalibacterium sp900539885"
# ]
    ### END EXPERIMENT
    

    # # 3. Display the result
    # print("\n--- Final Combined Microbiome ---")
    # print(subject_predicted_mb)
    # print(subject_predicted_mb.shape)

    ## Normalize by row.

    # # Step 0: Convert to normal scale
    # subject_predicted_mb_normal = 10 ** subject_predicted_mb

    # 1. Create a boolean mask to identify values not equal to the floor value.
    mask = subject_predicted_mb != 0.0001

    # 2. Calculate the sum of ONLY the non-floor values.
    # For a Series, .sum() doesn't need an axis.
    non_floor_sum = subject_predicted_mb[mask].sum()

    # 3. Update only the non-floor values by dividing them by their sum.
    # Using .loc[mask] is a direct way to modify a slice of the Series.
    subject_predicted_mb.loc[mask] = subject_predicted_mb[mask] / non_floor_sum

    # print("--- After normalization ---")
    # print(subject_predicted_mb)
    # The sum of the non-floor values will now be 1.0
    # The total sum will be 1.0 + (number of floor values * 0.0001)
    # print(f"Predicted Sum: {subject_predicted_mb.sum():.4f}")
    # print(f"Baseline Sum: {subject_baseline_mb.sum():.4f}")

    subject_ages.append(subject_age)
    subject_genders.append(subject_gender)

    all_mb_predictions.append(subject_predicted_mb)
    ####

    if recommendation is not None:

        original_total_kcal = recommendation['Current Diet (kcal)'].sum()

        # Keep only recommendations with a relative change of at least 50% of the food's original calorie share.
        min_rel_change = 0.5
        change_mask = (
            (recommendation['Current Diet (% Calorie)'] > 0) &
            (recommendation['Recommended Change (% Calorie)'].abs() >=
             min_rel_change * recommendation['Current Diet (% Calorie)'])
        )
        recommendation = recommendation.loc[change_mask].copy()

        # Compute the updated total after filtering
        delta_kcal = recommendation['Recommended Change (kcal)'].sum()
        new_total_kcal = original_total_kcal + delta_kcal

        recommendation['Subject'] = i
        recommendation['Original Total kcal'] = original_total_kcal
        recommendation['New Total kcal'] = new_total_kcal
        all_recommendations.append(recommendation)
        microbial_shifts.append(delta_species_df)
        subject_abundances[i] = subject_mb  # store current abundance

        for food, row in recommendation.iterrows():
            current_val = row['Current Diet (% Calorie)']
            change_val = row['Recommended Change (% Calorie)']

            if current_val > 0 and abs(change_val) >= min_rel_change * current_val:
                food_change_records[food].append({
                    'Subject': i,
                    'Food': food,
                    'Recommended Change (%)': change_val * 100,
                    'Recommended Change (kcal)': row['Recommended Change (kcal)']
                })

microbial_df = pd.concat(microbial_shifts, ignore_index=True)

###
predicted_mb_df = pd.DataFrame(all_mb_predictions)
# Display the final DataFrame
print("ALL PREDICTED MICROBIOMES")
# print(predicted_mb_df)
# predicted_mb_df.to_pickle(home_path + f"data/{SPECIES}/intervention_predicted_mb.pkl")

flat_records = [item for sublist in food_change_records.values() for item in sublist]
food_change_df = pd.DataFrame(flat_records)

if not microbial_df.empty:
    # Compute absolute FC
    microbial_df['fold_change'] = microbial_df['Predicted Δ fold Abundance'].abs()

    # # Get top 15 species by most consistently affected microbes
    # top_species = (
    #     microbial_df[microbial_df['Predicted Δ fold Abundance'] != 0]['Species']
    #     .value_counts()
    #     .head(15)
    #     .index
    # )

    # Filter to top species
    filtered_df = microbial_df[microbial_df['Species'].isin(triglyceride_species)].copy()
    print(filtered_df)

    # Compute frequency counts among top species only
    species_counts = (
        filtered_df['Species']
        .value_counts()
        .sort_values(ascending=False)
    )

    # Reorder filtered_df by frequency
    filtered_df['Species'] = pd.Categorical(
        filtered_df['Species'],
        categories=triglyceride_species,
        ordered=True
    )

    # Add species labels with frequency counts
    # Add species labels with actual counts
    label_map = (
        filtered_df['Species']
        .value_counts()
        .reindex(triglyceride_species)
        .fillna(0)
        .astype(int)
        .to_dict()
    )

    # Generate label strings
    label_map = {sp: f"{sp} (n={cnt})" for sp, cnt in label_map.items()}
    filtered_df['Species_Label'] = filtered_df['Species'].map(label_map)

    # Preserve plotting order
    ordered_labels = [label_map[sp] for sp in triglyceride_species if sp in label_map]

    # Plot
    plt.figure(figsize=(12, 8), dpi=300)
    sns.boxplot(
        data=filtered_df,
        y="Species_Label",
        x="Predicted Δ fold Abundance",
        order=ordered_labels,
        width=0.5,
        showcaps=True,
        boxprops={'facecolor': 'lightgray', 'edgecolor': 'black'},
        medianprops={'color': 'black'},
        whiskerprops={'color': 'black'},
        fliersize=0
    )
    sns.stripplot(
        data=filtered_df,
        y="Species_Label",
        x="Predicted Δ fold Abundance",
        order=ordered_labels,
        size=4,
        jitter=True,
        alpha=0.8,
        color='blue'
    )
    plt.title("Top Consistently Affected Microbes", fontsize=16)
    plt.xlabel("Predicted Relative Abundance Fold Change", fontsize=16, labelpad=10)
    plt.xticks(fontsize=14)
    plt.axvline(x=1, linestyle='--', color='red', alpha=0.6)
    plt.yticks(fontsize=16)
    plt.ylabel("")
    plt.grid(axis='x', linestyle='--', alpha=0.3)
    plt.tight_layout()
    # plt.savefig(f"{figures_path}/top_microbes_by_fc_freq_sorted.png", dpi=300)
    plt.show()

In [ ]:
microbial_df

In [ ]:
all_recommendations

In [ ]:
# Plot
plt.figure(figsize=(12, 8), dpi=300)
sns.boxplot(
    data=filtered_df,
    y="Species_Label",
    x="Predicted Δ fold Abundance",
    order=ordered_labels,
    width=0.5,
    showcaps=True,
    boxprops={'facecolor': 'lightgray', 'edgecolor': 'black'},
    medianprops={'color': 'black'},
    whiskerprops={'color': 'black'},
    fliersize=0
)
sns.stripplot(
    data=filtered_df,
    y="Species_Label",
    x="Predicted Δ fold Abundance",
    order=ordered_labels,
    size=4,
    jitter=True,
    alpha=0.6,
    color='blue'
)
plt.title("Top Consistently Affected Microbes", fontsize=16)
plt.xlabel("Predicted Relative Abundance Fold Change", fontsize=16, labelpad=10)
plt.xticks(fontsize=14)
plt.xlim(0.6, 1.6)
plt.axvline(x=1, linestyle='--', color='red', alpha=0.6)
plt.yticks(fontsize=16)
plt.ylabel("")
plt.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
# plt.savefig(f"{figures_path}/top_microbes_by_fc_freq_sorted.png", dpi=300)
plt.show()

In [ ]:
# Count how many times each species had nonzero predicted change
species_counts = (
    microbial_df[microbial_df['Predicted Δ fold Abundance'] != 0]['Species']
    .value_counts()
)

# Create a list of labels like: "Faecalibacterium prausnitzii (n=12)"
species_labels_with_n = [
    f"{species} (n={count})" for species, count in species_counts.items()
]

species_labels_with_n

In [ ]:
subject_baseline_mb.sort_values().tail(10)

In [ ]:
delta_species_df.sort_values(by="Predicted Δ fold Abundance")

In [ ]:
def plot_subject_recommendation_options(subject_df, subj_id, n_foods_to_show=10):
    top = subject_df.loc[subject_df['Recommended Change (kcal)'].abs() > 0].copy()
    top['Food'] = top.index

    # Compute optimization contribution per food
    top['Optimization Contribution'] = top['Recommended Change (% Calorie)'] * top['Personalized Impact Score']
    top_sorted = top.sort_values(by='Optimization Contribution', key=abs, ascending=False).head(n_foods_to_show)

    original_total_kcal = int(subject_df['Original Total kcal'].iloc[0])
    new_total_kcal = int(subject_df['New Total kcal'].iloc[0])

    plt.figure(figsize=(10, 6), dpi=300)
    ax = plt.gca()
    x_pos = [0, 1]

    for _, row in top_sorted.iterrows():
        color = ax._get_lines.get_next_color()
        y_start = row['Current Diet (kcal)']
        y_end = row['New Diet (kcal)']
        food = row['Food']
        delta = y_end - y_start
        delta_str = f"+{delta:.0f}" if delta > 0 else f"{delta:.0f}"
        label = f"{food} ({delta_str} kcal)"

        # Circle at current kcal value
        # ax.plot(x_pos[0], y_start, marker='o', markersize=6, color=color, label="_nolegend_")

        # Arrow from current to new
        ax.annotate(
            "", 
            xy=(x_pos[1], y_end), 
            xytext=(x_pos[0], y_start),
            arrowprops=dict(
                arrowstyle='-|>',         # clean shaft with triangle head
                color=color,
                lw=2.5,
                mutation_scale=14        # ← make arrowhead larger only
            ),
        )

        # Dummy line for legend
        ax.plot([], [], color=color, label=label, lw=2.5)

    # Set Y-axis limits with padding
    ymin = top_sorted[['Current Diet (kcal)', 'New Diet (kcal)']].min().min()
    ymax = top_sorted[['Current Diet (kcal)', 'New Diet (kcal)']].max().max()
    ax.set_ylim(ymin * 0.9, ymax * 1.1)

    # Axes setup
    plt.xticks(ticks=x_pos, labels=['Current', 'New'], fontsize=14)
    plt.title(f"Example Individual: Suggested Food Adjustments Subject {subj_id}", fontsize=15)
    plt.ylabel("Kilocalories / Day", fontsize=16, labelpad=7)
    plt.xlabel("Diet", fontsize=16)
    plt.yticks(fontsize=14)

    # Legend
    calories_text = f"Total kcal: {original_total_kcal} → {new_total_kcal}"
    plt.legend(title=calories_text, title_fontsize=16, bbox_to_anchor=(1, 1), loc='upper left', fontsize=14)
    plt.tight_layout()
    # plt.savefig(f"{figures_path}/subject_{subj_id}_recommendation.png", dpi=300)
    # plt.close()
    plt.show()

# Run for subject
for subj_id in range(10):
    subj_df = next((rec for rec in all_recommendations if rec['Subject'].iloc[0] == subj_id), None)
    if subj_df is not None:
        plot_subject_recommendation_options(subj_df, subj_id)

Case study for bacteria change

In [ ]:
# def plot_subject_microbiome_shift(microbial_df, subj_id, current_species_abundance, n_species_to_show=10):
#     subj_species = microbial_df[microbial_df['Subject'] == subj_id].copy()
#     subj_species['Abs Change'] = subj_species['Predicted Δ Abundance'].abs()
#     subj_species = subj_species.sort_values(by='Abs Change', ascending=False).head(n_species_to_show)

#     if subj_species.empty:
#         print(f"No microbial shift data for subject {subj_id}")
#         return
#     # Add baseline abundance and compute new abundance
#     subj_species['Baseline Abundance'] = subj_species['Species'].map(current_species_abundance)
#     subj_species.dropna(subset=['Baseline Abundance'], inplace=True)

#     # Convert baseline and delta to % points first
#     subj_species['Baseline Abundance'] *= 100
#     subj_species['Predicted Δ Abundance'] *= 100

#     # Then compute predicted abundance in % points
#     subj_species['Predicted Abundance'] = subj_species['Baseline Abundance'] + subj_species['Predicted Δ Abundance']

#     plt.figure(figsize=(10, 6), dpi=300)
#     ax = plt.gca()
#     x_pos = [0, 1]

#     for _, row in subj_species.iterrows():
#         color = ax._get_lines.get_next_color()
#         y_start = row['Baseline Abundance']
#         y_end = row['Predicted Abundance']
#         delta = y_end - y_start
#         label = f"{row['Species']} ({delta:+.2f} % pts)"

#         # Dot at baseline
#         ax.plot(x_pos[0], y_start, marker='o', markersize=6, color=color, label="_nolegend_")

#         # Arrow to predicted abundance
#         ax.annotate(
#             "",
#             xy=(x_pos[1], y_end),
#             xytext=(x_pos[0], y_start),
#             arrowprops=dict(
#                 arrowstyle='-|>',
#                 color=color,
#                 lw=2.5,
#                 mutation_scale=14
#             )
#         )

#         # Dummy line for legend
#         ax.plot([], [], color=color, label=label, lw=2.5)

#     # Y-limits with padding
#     y_min = min(subj_species['Baseline Abundance'].min(), subj_species['Predicted Abundance'].min())
#     y_max = max(subj_species['Baseline Abundance'].max(), subj_species['Predicted Abundance'].max())
#     padding = (y_max - y_min) * 0.1 or 0.01
#     ax.set_ylim(y_min - padding, y_max + padding)

#     # Axes
#     plt.xticks(ticks=x_pos, labels=['Baseline', 'Predicted'], fontsize=14)
#     plt.ylabel("Abundance (% points)", fontsize=16)
#     plt.xlabel("Microbiome", fontsize=16)
#     plt.title(f"Subject {subj_id}: Microbial Abundance Shifts", fontsize=15)
#     plt.yticks(fontsize=14)

#     # Legend
#     plt.legend(title="Predicted Shifts", title_fontsize=11, bbox_to_anchor=(1, 1), loc='upper left', fontsize=10)
#     plt.tight_layout()
#     plt.show()


# for delta_species_df in microbial_shifts[1:10]:
#     subj_id = delta_species_df['Subject'].iloc[0]
#     current_abundance = subject_abundances[subj_id]
#     plot_subject_microbiome_shift(delta_species_df, subj_id, current_abundance)


Copy of most updated code, for plotting only

In [ ]:
microbial_shifts

In [ ]:
microbial_shifts

In [ ]:
def plot_subject_microbiome_shift(microbial_df, subj_id, n_species_to_show=10):
    # Filter for this subject's predicted species shifts
    subj_species = microbial_df[microbial_df['Subject'] == subj_id].copy()

    if subj_species.empty:
        print(f"No microbial shift data for subject {subj_id}")
        return

    # Sort by absolute change and select top N
    subj_species['Abs Change'] = subj_species['Predicted Δ linear Abundance'].abs()
    subj_species = subj_species.sort_values(by='Abs Change', ascending=False).head(n_species_to_show)

    # Convert all values to % points (from [0, 1] scale)
    for col in ['Baseline Abundance', 'Predicted Δ linear Abundance', 'Predicted Abundance']:
        if subj_species[col].max() < 1.5:  # Assuming values are in [0,1] if unconverted
            subj_species[col] *= 100

    # Plot setup
    plt.figure(figsize=(10, 6), dpi=300)
    ax = plt.gca()
    x_pos = [0, 1]

    for _, row in subj_species.iterrows():
        color = ax._get_lines.get_next_color()
        y_start = row['Baseline Abundance']
        y_end = row['Predicted Abundance']
        delta = y_end - y_start
        label = f"{row['Species']} ({delta:+.2f}%)"

        # Dot at baseline
        # ax.plot(x_pos[0], y_start, marker='o', markersize=6, color=color, label="_nolegend_")

        # Arrow to predicted abundance
        ax.annotate(
            "",
            xy=(x_pos[1], y_end),
            xytext=(x_pos[0], y_start),
            arrowprops=dict(
                arrowstyle='-|>',
                color=color,
                lw=2.5,
                mutation_scale=14
            )
        )

        # Dummy line for legend
        ax.plot([], [], color=color, label=label, lw=2.5)

    # Set Y limits with padding
    y_min = min(subj_species['Baseline Abundance'].min(), subj_species['Predicted Abundance'].min())
    y_max = max(subj_species['Baseline Abundance'].max(), subj_species['Predicted Abundance'].max())
    padding = (y_max - y_min) * 0.1 or 0.01
    ax.set_ylim(y_min - padding, y_max + padding)

    # Axes formatting
    plt.xticks(ticks=x_pos, labels=['Baseline', 'Predicted'], fontsize=14)
    plt.ylabel("Relative Abundance (%)", fontsize=16)
    plt.xlabel("Microbiome", fontsize=16)
    plt.title(f"Subject {subj_id}: Microbial Abundance Shifts", fontsize=15)
    plt.yticks(fontsize=14)

    # Legend
    plt.legend(title="Predicted Shifts", title_fontsize=12, bbox_to_anchor=(1, 1), loc='upper left', fontsize=11)
    plt.tight_layout()
    plt.show()

for delta_species_df in microbial_shifts[0:25]:
    subj_id = delta_species_df['Subject'].iloc[0]
    plot_subject_microbiome_shift(delta_species_df, subj_id)


### Predicting Triglycerides

In [ ]:
models_dict = pickle.load(open(f'/net/mraid20/export/genie/LabData/Analyses/tomerse/diet_mb/models/{SPECIES}/models_mb_phenotypes.pkl', 'rb'))
models_dict.keys()

In [ ]:
def create_scaled_df(df_features, age_data, sex_data, targets, scaler_path):

    # 1. Load the scalers from the specified path
    with open(f'{scaler_path}/mb_scaler.pkl', 'rb') as f:
        species_scaler = pickle.load(f)
    with open(f'{scaler_path}/age_scaler.pkl', 'rb') as f:
        age_scaler = pickle.load(f)

    # 2. Scale the microbiome features and create a new DataFrame
    scaled_mb = species_scaler.transform(df_features[targets])
    scaled_df = pd.DataFrame(scaled_mb, index=df_features.index, columns=targets)

    # 3. Add scaled age and sex columns
    # Ensure age_data is a 2D array for the scaler
    scaled_df['age'] = age_scaler.transform(pd.Series(age_data).values.reshape(-1, 1))
    # Assign sex data, using .values to prevent potential index mismatch issues
    scaled_df['sex'] = sex_data

    # 4. Reorder the columns to the desired format
    new_order = ['age', 'sex'] + targets
    final_df = scaled_df[new_order]

    return final_df

In [ ]:
predicted_mb_df

In [ ]:
# predicted_mb_df_experimental = 
predicted_mb_df["Bifidobacterium longum"]

In [ ]:
predicted_mb_df.columns = targets
predicted_mb_df = np.log10(predicted_mb_df)
predicted_mb_df

In [ ]:
scaler_directory = home_path + f'data/{SPECIES}'

predicted_mb_df_scaled = create_scaled_df(
    df_features=predicted_mb_df,
    age_data=subject_ages,
    sex_data=subject_genders,
    targets=targets,
    scaler_path=scaler_directory
)

print("Scaled Predicted DataFrame description:")
predicted_mb_df_scaled.iloc[:, 0:7].describe()

In [ ]:
baseline_mb_df = diet_mb[['age', 'sex'] + targets].loc[predicted_mb_df.index]

baseline_mb_df_scaled = create_scaled_df(
    df_features=baseline_mb_df,
    age_data=baseline_mb_df['age'],
    sex_data=baseline_mb_df['sex'],
    targets=targets,
    scaler_path=scaler_directory
)

# Display the result
print("\nScaled Baseline DataFrame:")
baseline_mb_df_scaled

In [ ]:
# 1. Compute the difference
diff = predicted_mb_df_scaled - baseline_mb_df_scaled

# 2. Mask out non‐positive values (they’ll become NaN)
pos_only = diff.where(diff > 0)

# 3a. If you just want to see a Series of (row, col)→value for all positives:
positive_series = pos_only.stack()
print(positive_series.sort_values(ascending=False))



In [ ]:
predicted_mb_df.loc[:, 'fBin__369|gBin__1495|sBin__2222']

In [ ]:
predicted_mb_df_scaled.loc[:, 'fBin__369|gBin__1495|sBin__2222']

In [ ]:
def get_oof_preds(X, target):
    model = models_dict[target]
    y_pred = model.predict(X)
    return y_pred


baseline_trig = get_oof_preds(baseline_mb_df_scaled, 'bt__triglycerides')
baseline_trig

In [ ]:
intervention_trig = get_oof_preds(predicted_mb_df_scaled, 'bt__triglycerides')
intervention_trig

In [ ]:
intervention_trig - baseline_trig

In [ ]:
predicted_mb_df_scaled.index[9]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Convert to pandas Series and compute difference
diff = pd.Series(intervention_trig - baseline_trig)

# Plot
plt.figure(figsize=(8, 5), dpi=300)
sns.histplot(diff, bins=30, kde=True, color='skyblue')
plt.axvline(0, color='black', linestyle='--', label='No Change')
plt.title('Distribution of Triglyceride Change (Intervention - Baseline)')
plt.xlabel('Δ Triglycerides (mg/dL)')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# import pandas as pd

# # Convert to pandas Series and compute difference
# diff_ratio = pd.Series(intervention_trig / baseline_trig)

# # Plot
# plt.figure(figsize=(8, 5), dpi=300)
# sns.histplot(diff_ratio, bins=30, kde=True, color='skyblue')
# plt.axvline(1, color='black', linestyle='--', label='No Change')
# plt.title('Distribution of Triglyceride Change (Intervention - Baseline)')
# plt.xlabel('Δ Triglycerides Ratio')
# plt.xlim(0.8, 1.2)
# plt.ylabel('Count')
# plt.legend()
# plt.tight_layout()
# plt.show()

In [ ]:
food_impact_scores_list[0]

In [ ]:
import pandas as pd
from scipy.stats import spearmanr, pearsonr

# For each subject: store SHAP objective and predicted TG delta
results_df = pd.DataFrame({
    "LP_Objective": [-fs.sum() for fs in food_impact_scores_list],  # Negative because LP minimizes
    "Δ_TG_Model": intervention_trig - baseline_trig
})

# Correlation
pearson_corr, _ = pearsonr(results_df["LP_Objective"], results_df["Δ_TG_Model"])
spearman_corr, _ = spearmanr(results_df["LP_Objective"], results_df["Δ_TG_Model"])

print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"Spearman correlation: {spearman_corr:.3f}")
